In [1]:
import os
import json
import warnings
import sys
import time
warnings.filterwarnings('ignore')

from tqdm import tqdm
from pathlib import Path
from collections import Counter, defaultdict
from PIL import Image

In [2]:
try:
    import google.colab
    from google.colab import drive
    !uv pip install open-clip-torch
    drive.mount('/content/drive', force_remount=True)
    PROJECT_ROOT = Path('/content/drive/Othercomputers/my_notebook/lion_final_pro_multimodal-anomaly-report-generation') # 본인 경로 수정: Mac/Window
except ImportError:
    PROJECT_ROOT = Path.cwd().parents[1]

os.chdir(PROJECT_ROOT) # 현재 경로 수정
print(f"Current working directory: {os.getcwd()}")

Using Python 3.12.12 environment at: /usr
Resolved 49 packages in 434ms
Prepared 2 packages in 85ms
Installed 2 packages in 15ms
 + ftfy==6.3.1
 + open-clip-torch==3.2.0
Mounted at /content/drive
Current working directory: /content/drive/Othercomputers/my_notebook/lion_final_pro_multimodal-anomaly-report-generation


In [3]:
import torch
import numpy as np
from PIL import Image
import torchvision.transforms as T
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path
from tqdm import tqdm
import pickle

class DinoV2Retrieval:
    def __init__(self, model_type='dinov2_vitg14', device='cuda'):
        # model_type: 'dinov2_vits14' (빠름), 'dinov2_vitb14', 'dinov2_vitl14', 'dinov2_vitg14' (정확)
        self.device = device
        self.model = torch.hub.load('facebookresearch/dinov2', model_type).to(device)
        self.model.eval()

        # DINOv2 공식 전처리 설정
        self.transform = T.Compose([
            T.Resize(256),
            T.CenterCrop(224),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        self.train_features = []
        self.train_paths = []

    @torch.no_grad()
    def get_embedding(self, image_path):
        img = Image.open(image_path).convert('RGB')
        img_t = self.transform(img).unsqueeze(0).to(self.device)
        embedding = self.model(img_t)
        return embedding.cpu().numpy()

    def build_index(self, train_dir):
        """Train 폴더 내의 모든 정상 이미지를 벡터화하여 메모리에 저장"""
        self.train_features = []
        self.train_paths = []

        # 이미지 확장자 필터링
        extensions = ('.png', '.jpg', '.jpeg', '.bmp')
        img_paths = [p for p in Path(train_dir).rglob('*') if p.suffix.lower() in extensions]

        print(f"인덱싱 시작: {len(img_paths)}개 이미지")
        for p in tqdm(img_paths):
            feat = self.get_embedding(str(p))
            self.train_features.append(feat)
            self.train_paths.append(str(p))

        self.train_features = np.vstack(self.train_features)
        print("✅ 인덱싱 완료!")

    def search(self, test_img_path, top_k=1):
        """가장 유사한 이미지 찾기"""
        test_feat = self.get_embedding(test_img_path)

        # 코사인 유사도 계산
        similarities = cosine_similarity(test_feat, self.train_features)[0]
        top_indices = np.argsort(similarities)[::-1][:top_k]

        results = []
        for idx in top_indices:
            results.append({
                "path": self.train_paths[idx],
                "score": float(similarities[idx])
            })
        return results

    def save_index(self, save_path):
        """인덱싱된 데이터를 파일로 저장"""
        data = {
            "features": self.train_features,
            "paths": self.train_paths
        }
        with open(save_path, 'wb') as f:
            pickle.dump(data, f)
        print(f"✅ 인덱스가 {save_path}에 저장되었습니다.")

    def load_index(self, load_path):
        """파일에서 인덱싱된 데이터를 불러오기"""
        with open(load_path, 'rb') as f:
            data = pickle.load(f)
        self.train_features = data["features"]
        self.train_paths = data["paths"]
        print(f"✅ {len(self.train_paths)}개의 인덱스를 성공적으로 불러왔습니다.")


In [12]:
# --- 실행 예시 ---
# 1. 초기화
PROJECT_ROOT = Path('/content/drive/Othercomputers/my_notebook/lion_final_pro_multimodal-anomaly-report-generation')
BASE_GOODS_DIR = os.path.join(PROJECT_ROOT, "dataset/MMAD/GoodsAD")
SAVE_DIR = os.path.join(PROJECT_ROOT, "results/dino_rag")
os.makedirs(SAVE_DIR, exist_ok=True)

retriever = DinoV2Retrieval(model_type='dinov2_vits14')

# 2. 하위의 모든 클래스 폴더 리스트업
class_list = [d for d in os.listdir(BASE_GOODS_DIR)
              if os.path.isdir(os.path.join(BASE_GOODS_DIR, d)) and not d.startswith('.')]

print(f"🚀 총 {len(class_list)}개의 클래스를 인덱싱합니다: {class_list}")

# 4. 루프를 돌며 클래스별로 pkl 생성
for class_name in class_list:
    print(f"\n--- [{class_name}] 작업 시작 ---")

    # 해당 클래스의 train/good 경로 설정
    train_good_dir = os.path.join(BASE_GOODS_DIR, class_name, "train", "good")

    # 경로가 존재하는지 확인
    if not os.path.exists(train_good_dir):
        print(f"❌ 경로를 찾을 수 없어 건너뜁니다: {train_good_dir}")
        continue

    # 해당 클래스 이미지들로 인덱스 빌드
    retriever.build_index(train_good_dir)

    # 클래스 이름을 파일명으로 하여 저장 (예: pushpins.pkl)
    save_path = os.path.join(SAVE_DIR, f"{class_name}.pkl")
    retriever.save_index(save_path)


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


인덱싱 시작: 183개 이미지


100%|██████████| 183/183 [00:33<00:00,  5.45it/s]

✅ 인덱싱 완료!
✅ 인덱스가 /content/drive/Othercomputers/my_notebook/lion_final_pro_multimodal-anomaly-report-generation/results/dino_rag/cigarette_box_index.pkl에 저장되었습니다.


In [17]:
# 1. 인덱스 불러오기
retriever.load_index(PROJECT_ROOT / "results/dino_rag/cigarette_box_index.pkl")

# 2. 바로 검색
test_img = PROJECT_ROOT / "dataset/MMAD/GoodsAD/cigarette_box/test/opened/001_006.jpg"
matches = retriever.search(str(test_img), top_k=3)

for res in matches:
    print(f"유사 이미지: {res['path']} (점수: {res['score']:.4f})")

✅ 183개의 인덱스를 성공적으로 불러왔습니다.
유사 이미지: /content/drive/Othercomputers/my_notebook/lion_final_pro_multimodal-anomaly-report-generation/dataset/MMAD/GoodsAD/cigarette_box/train/good/001_000.jpg (점수: 0.8978)
유사 이미지: /content/drive/Othercomputers/my_notebook/lion_final_pro_multimodal-anomaly-report-generation/dataset/MMAD/GoodsAD/cigarette_box/train/good/001_002.jpg (점수: 0.8695)
유사 이미지: /content/drive/Othercomputers/my_notebook/lion_final_pro_multimodal-anomaly-report-generation/dataset/MMAD/GoodsAD/cigarette_box/train/good/036_002.jpg (점수: 0.7852)


Class Clasifier

In [ ]:
import torch
import torchvision.transforms as T
from PIL import Image
import pickle
import os
import numpy as np
import matplotlib.pyplot as plt

# 1. 모델 로드 (VITS14 모델이 가볍고 빠릅니다)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dinov2 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(device)
dinov2.eval()

# 2. 전처리 함수
def get_transform():
    return T.Compose([
        T.Resize(256),
        T.CenterCrop(224),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

# 3. 분산된 .pkl 파일 통합 로더
def load_and_integrate_pkl(pkl_folder):
    combined_vectors = []
    combined_labels = []
    combined_paths = []

    pkl_files = [f for f in os.listdir(pkl_folder) if f.endswith('.pkl')]
    print(f"📦 총 {len(pkl_files)}개의 클래스 파일을 발견했습니다.")

    for file_name in pkl_files:
        with open(os.path.join(pkl_folder, file_name), 'rb') as f:
            data = pickle.load(f)
            # 파일명을 클래스 이름으로 사용 (예: pushpins.pkl -> pushpins)
            class_name = file_name.replace('.pkl', '')

            combined_vectors.append(data['vectors'])
            combined_labels.extend([class_name] * len(data['vectors']))
            combined_paths.extend(data['paths'])

    return {
        "vectors": np.vstack(combined_vectors),
        "labels": np.array(combined_labels),
        "paths": np.array(combined_paths)
    }

# 4. 분류기 메인 함수
def classify_image(image_path, index, top_k=1):
    # 이미지 특징 추출
    img = Image.open(image_path).convert('RGB')
    transform = get_transform()
    img_tensor = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        query_vector = dinov2(img_tensor).cpu().numpy()

    # 코사인 유사도 계산 (벡터 정규화 후 내적)
    # norm을 계산하여 방향만 비교합니다.
    dist = np.linalg.norm(index['vectors'] - query_vector, axis=1)

    # 가장 가까운 인덱스 순서대로 정렬
    nearest_indices = np.argsort(dist)[:top_k]

    results = []
    for idx in nearest_indices:
        results.append({
            "category": index['labels'][idx],
            "distance": dist[idx],
            "ref_path": index['paths'][idx]
        })

    return results[0] # Top-1 결과 반환

# --- [테스트 실행 구문] ---
# 1. pkl 파일들이 모여있는 폴더 경로 지정
PKL_DIR = "/content/drive/MyDrive/my_pkl_folder"

# 2. 통합 인덱스 생성
integrated_index = load_and_integrate_pkl(PKL_DIR)

# 3. 테스트할 이미지 경로
test_img = "test_sample.jpg"

# 4. 결과 출력
prediction = classify_image(test_img, integrated_index)

print("\n" + "="*30)
print(f"🎯 최종 예측 클래스: {prediction['category']}")
print(f"📏 거리(Distance): {prediction['distance']:.4f}")
print(f"🖼️ 참조 이미지: {prediction['ref_path']}")
print("="*30)